In [ ]:
import os
import glob
import utils
import numpy as np
import utils
import importlib
import xarray as xr

importlib.reload(utils)

print("starting script", flush=True)
base_path = "../data/rsds_past"

files_past = []
files_future = []

# Walk through directory structure to collect file paths
for root, dirs, files in os.walk(base_path):
    if not dirs:
        rel_path = os.path.relpath(root, base_path)
        path_past = os.path.join(base_path, rel_path)
        future_path = path_past.replace("rsds_past", "rsds_future")

        # Find nc files
        past_files = glob.glob(os.path.join(path_past, "*r1i1p1_1995*.nc"))
        future_files = glob.glob(os.path.join(future_path, "*r1i1p1_2045*.nc"))

        files_past.extend(past_files)
        files_future.extend(future_files)


# Use the file names as model identifiers
model_names_past = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_past
]
model_names_future = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_future
]

print("File source loaded", flush=True)
# Select time frame
files_raw_past = utils.pre_process(files_past, ["rsds"], 0, 5, 0, 5)
files_past = utils.select_time_frame(files_raw_past, slice("1995-01-01", "2004-12-30"))

files_raw_future = utils.pre_process(files_future, ["rsds"], 0, 5, 0, 5)
files_future = utils.select_time_frame(
    files_raw_future, slice("2045-01-01", "2054-12-30")
)
print("Files loaded", flush=True)

In [ ]:
def marginal_average_seasonal_hourly(
    files, files_future, model_names, model_names_future
):
    """
    Computes seasonal mean per day and calculates change between past and future. This is only applied to the marginals of solar radiation.
    """
    import numpy as np

    seasons = ["DJF", "MAM", "JJA", "SON"]
    DMPE_diff = []

    for i, temp_data in enumerate(files):
        seasonal_dmpe = {}
        index = model_names_future.index(model_names[i])
        reference = files_future[index]

        for season in seasons:
            # Select seasonal data
            temp_season = temp_data.sel(time=temp_data["time.season"] == season)
            ref_season = reference.sel(time=reference["time.season"] == season)

            if temp_season.size == 0 or ref_season.size == 0:
                continue  # skip empty seasons

            # Group by hour and average over time for each hour
            temp_hourly = temp_season.mean(dim="time")
            ref_hourly = ref_season.mean(dim="time")

            # DMPE per hour and grid cell
            with np.errstate(divide="ignore", invalid="ignore"):
                dmpe_hourly = ref_hourly - temp_hourly
            seasonal_dmpe[season] = dmpe_hourly  # dims: lat x lon

        DMPE_diff.append(seasonal_dmpe)

    return DMPE_diff

In [ ]:
marginal = marginal_average_seasonal_hourly(
    files_past, files_future, model_names_past, model_names_future
)

In [ ]:
seasons = ["DJF", "MAM", "JJA", "SON"]
# Stack all models
model_arrays = []
for m in range(len(marginal)):
    season_dict = marginal[m]  # {'DJF': <DataArray>, ...}
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# Concatenate along model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

# numpy array: shape (model, season, lat, lon)
array = all_models_array.values
np.save(
    "../plotting_data/rsds_spatial/marginal_average_seasonal_hourly_abs.npy",
    array,
)

In [ ]:
marginal = utils.marginal_var_seasonal(
    files_past, files_future, model_names_past, model_names_future
)

# Stack all models
model_arrays = []
for m in range(len(marginal)):
    season_dict = marginal[m]
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# Concatenate along model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

# numpy array: shape (model, season, lat, lon)
array = all_models_array.values
np.save(
    "../plotting_data/rsds_spatial/marginal_var_seasonal.npy",
    array,
)

In [ ]:
marginal = utils.spatiotemporal_below_percentile_seasonal(
    files_past, files_future, model_names_past, model_names_future
)


seasons = ["DJF", "MAM", "JJA", "SON"]
# Stack all models
model_arrays = []
for m in range(len(marginal)):
    season_dict = marginal[m]
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# Concatenate along model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

# numpy array: shape (model, season, lat, lon)
array = all_models_array.values
np.save(
    "../plotting_data/rsds_spatial/spatiotemporal_below_percentile_seasonal.npy",
    array,
)

In [ ]:
marginal = utils.spatiotemporal_below_percentile_seasonal(
    files_past, files_future, model_names_past, model_names_future, 50
)


seasons = ["DJF", "MAM", "JJA", "SON"]
# stack all models
model_arrays = []
for m in range(len(marginal)):
    season_dict = marginal[m]
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# concatenate along model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

# numpy array: shape (model, season, lat, lon)
array = all_models_array.values
np.save(
    "../plotting_data/rsds_spatial/spatiotemporal_above_percentile_seasonal.npy",
    array,
)

In [ ]:
files_past = []
files_future = []

for root, dirs, files in os.walk(base_path):
    if not dirs:
        rel_path = os.path.relpath(root, base_path)
        path_past = os.path.join(base_path, rel_path)
        future_path = path_past.replace("rsds_past", "rsds_future")

        # Find nc files
        past_files = glob.glob(os.path.join(path_past, "*r1i1p1_1995*.nc"))
        future_files = glob.glob(os.path.join(future_path, "*r1i1p1_2045*.nc"))

        files_past.extend(past_files)
        files_future.extend(future_files)


# Use the file names as model identifiers
model_names_past = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_past
]
model_names_future = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_future
]


files_raw_past = utils.pre_process(files_past[0:1], ["rsds"], 0, 5, 0, 5)
print(files_raw_past)
files_past = utils.select_time_frame(files_raw_past, slice("1995-01-01", "2004-12-30"))

# Compute relative variability (RCM / GCM)

# Coordinates
lons = files_past[0].coords["lon"].values
lats = files_past[0].coords["lat"].values

import os

# Define and create the directory
output_dir = "../plotting_data/rsds_spatial/"
os.makedirs(output_dir, exist_ok=True)

# 1. Save Lons and Lats (as plain text, one value per line)
np.savetxt(os.path.join(output_dir, "lons.txt"), lons, delimiter="\n")
np.savetxt(os.path.join(output_dir, "lats.txt"), lats, delimiter="\n")

# 2. Save Model Names (one name per line)
with open(os.path.join(output_dir, "model_names_past.txt"), "w") as f:
    f.write("\n".join(model_names_past))

with open(os.path.join(output_dir, "model_names_future.txt"), "w") as f:
    f.write("\n".join(model_names_future))

print(f"Files saved successfully in {output_dir}")